# Understand RAG Fundamentals

In [16]:
# load llm 
# Create environment
import os 
from dotenv import load_dotenv 
load_dotenv()
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY") 
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")


# Get LLM model 
from langchain.chat_models import init_chat_model 

primary_llm = init_chat_model(model="llama-3.3-70b-versatile", model_provider="Groq")

fallback_llm_1 = init_chat_model(model="gpt-5.4-nano", model_provider="openai",
                 model_kwargs={"temperature": 0.5, "max_tokens": 1000})
fallback_llm_2 = init_chat_model(model="gpt-5.4-mini", model_provider="openai",
                 model_kwargs={"temperature": 0.5, "max_tokens": 1000})

/Users/nali/Documents/YTLLMs/.venv/lib/python3.13/site-packages/langchain/chat_models/base.py:496: UserWarning: Parameters {'temperature', 'max_tokens'} should be specified explicitly. Instead they were passed in as part of `model_kwargs` parameter.
  return _init_chat_model_helper(


## Select documents for RAG application

In [17]:
import requests
from langchain_core.documents import Document

from langchain_core.vectorstores import InMemoryVectorStore

from langchain_openai import OpenAIEmbeddings

from langchain_text_splitters import RecursiveCharacterTextSplitter

DOCS_BASE = "https://docs.langchain.com"

# Curated LangChain OSS pages for this tutorial. Expand this list or parse
# URLs from https://docs.langchain.com/llms.txt to index more of the site.
DOC_PATHS = [
    "oss/python/langchain/agents",
    "oss/python/deepagents/rag",
    "oss/python/langchain/tools",
    "oss/python/langchain/models",
    "oss/python/deepagents/retrieval",
    "oss/python/langchain/knowledge-base",
    "oss/python/langchain/middleware",
    "oss/python/deepagents/overview",
    "oss/python/deepagents/subagents",
    "oss/python/deepagents/streaming",
    "oss/python/deepagents/frontend/subagent-streaming",
    "oss/python/deepagents/backends",
    "oss/python/langgraph/overview",
    "oss/python/langgraph/quickstart",
]

## Load documents for RAG application

In [18]:
def load_langchain_docs(doc_paths: list[str] | None = None) -> list[Document]:
    """Fetch LangChain documentation pages as Documents."""
    paths = doc_paths or DOC_PATHS
    docs: list[Document] = []
    for path in paths:
        url = f"{DOCS_BASE}/{path}.md"
        try:
            response = requests.get(url, timeout=20)
            response.raise_for_status()
        except requests.RequestException:
            continue
        source = f"{DOCS_BASE}/{path}"
        docs.append(
            Document(page_content=response.text, metadata={"source": source})
        )
    return docs


docs = load_langchain_docs()
print(f"Loaded {len(docs)} documentation pages.")

Loaded 14 documentation pages.


## Split the documents

In [19]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
all_splits = text_splitter.split_documents(docs)
print(f"No. of splits: {len(all_splits)}")

No. of splits: 896


## Add your embedding model 

In [20]:
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-mpnet-base-v2",
    encode_kwargs={"normalize_embeddings": True,
            'batch_size':32
                   },
    
)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 21944.70it/s]


In [21]:
vector = embeddings.embed_query("Langhcain agent creation")
print(len(vector))
print(vector)

768
[0.03305681422352791, -0.026300257071852684, -0.01862768642604351, 0.01701512560248375, -0.01132028829306364, -0.02466646023094654, 0.04203424230217934, -0.02621842734515667, 0.05911308899521828, 0.009283211082220078, 0.013977302238345146, -0.0012077896390110254, -0.022574206814169884, 0.017680997028946877, 0.0161933284252882, 0.0004457463219296187, 0.022729871794581413, -0.0241558738052845, 0.0020727820228785276, -0.031905509531497955, -0.008226610720157623, 0.011623810045421124, 0.04033198952674866, 0.03199523687362671, 0.0034876351710408926, -0.04886274039745331, -0.029822219163179398, 0.003514420473948121, 0.015942838042974472, 0.01985342986881733, -0.0442696139216423, -0.0421137809753418, 0.0337846577167511, 0.02349468134343624, 1.2140669696236728e-06, -0.052771780639886856, -0.02538640983402729, -0.01690710335969925, -0.05139066278934479, -0.025494111701846123, 0.056734614074230194, 0.01656297966837883, -0.03401315212249756, 0.009992377832531929, -0.0417645126581192, 0.033090

## Create a vector store 

In [22]:
from langchain_chroma import Chroma

vector_store = Chroma(
    collection_name="example_collection",
    embedding_function=embeddings,
    persist_directory="./chroma_langchain_db",  # Where to save data locally, remove if not necessary
)

In [23]:
vector_store.add_documents(documents=all_splits)
print(f"Indexed {len(all_splits)} chunks.")

Indexed 896 chunks.


### Do some Semantic Search testing 

In [24]:
vector_store.similarity_search(query="What is Langchain agent?",k=1)

[Document(id='8abf740c-7d86-42e6-a743-9c2fb99bee87', metadata={'source': 'https://docs.langchain.com/oss/python/langgraph/overview'}, page_content='<Expandable title="how LangChain products fit together" defaultOpen={false}>\n  * [Deep Agents](/oss/python/deepagents/overview) is an [agent harness](/oss/python/concepts/products#agent-harnesses-like-the-deep-agents-sdk): planning, subagents, filesystem tools, and context management on top of LangGraph.\n  * [LangChain](/oss/python/langchain/overview) is the agent framework: abstractions and integrations for models, tools, and agent loops.\n  * [LangGraph](/oss/python/langgraph/overview) is the orchestration runtime: durable execution, streaming, human-in-the-loop, and persistence.\n  * [LangSmith](/langsmith/observability) is the platform for tracing, evaluation, prompts, and deployment across frameworks.\n  * [LangSmith Engine](/langsmith/engine) detects issues in your LangGraph agent traces and proposes fixes. You can open a pull reque

In [25]:
results = vector_store.similarity_search_with_score(query="What is Langchain agent?",k=5)

for result in results: 
    doc, score = result
    print(score)
    print(doc.page_content[:100])


0.5610599517822266
<Expandable title="how LangChain products fit together" defaultOpen={false}>
  * [Deep Agents](/oss/
0.5610599517822266
<Expandable title="how LangChain products fit together" defaultOpen={false}>
  * [Deep Agents](/oss/
0.5610599517822266
<Expandable title="how LangChain products fit together" defaultOpen={false}>
  * [Deep Agents](/oss/
0.5610599517822266
<Expandable title="how LangChain products fit together" defaultOpen={false}>
  * [Deep Agents](/oss/
0.6833313703536987
[LangChain](/oss/python/langchain/) is the framework that provides the core building blocks for your


## Retriever interface 

In [26]:
retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={"k":2}
)

In [27]:
retriever.invoke(input="How to create Langgrapgh agents?")

[Document(id='ec410413-8ca8-4d66-a0e0-081e2a74d80a', metadata={'source': 'https://docs.langchain.com/oss/python/langgraph/quickstart'}, page_content='<Tip>\n      Trace and debug your agent with [LangSmith](https://smith.langchain.com?utm_source=docs\\&utm_medium=cta\\&utm_campaign=langsmith-signup\\&utm_content=oss-langgraph-quickstart). Follow the [tracing quickstart](/langsmith/trace-with-langgraph) to get set up. When ready for production, see [Deploy](/langsmith/deployment) for hosting options.\n\n      We recommend you also set up [LangSmith Engine](/langsmith/engine) which monitors your traces, detects issues, and proposes fixes.\n    </Tip>\n\n    Congratulations! You\'ve built your first agent using the LangGraph Functional API.\n\n    <Accordion title="Full code example" icon="code">\n      ```python theme={"theme":{"light":"catppuccin-latte","dark":"catppuccin-mocha"}}\n      # Step 1: Define tools and model\n\n      from langchain.tools import tool\n      from langchain.cha

In [28]:
retriever.batch([
    "What is an agent", "What is a deep agent"
])

[[Document(id='dd9babb0-5e45-4b3d-bc01-957cd882b758', metadata={'source': 'https://docs.langchain.com/oss/python/deepagents/overview'}, page_content="[LangChain](/oss/python/langchain/) is the framework that provides the core building blocks for your agents.\nTo learn more about the differences between LangChain, LangGraph, and Deep Agents, see [Frameworks, runtimes, and harnesses](/oss/python/concepts/products). For a side-by-side comparison with Anthropic's harness, see [Deep Agents vs. Claude Agent SDK](/oss/python/deepagents/comparison).\n\nFor building custom agents without these built-in capabilities, consider using LangChain's [`create_agent`](/oss/python/langchain/agents) or building a custom [LangGraph](/oss/python/langgraph/overview) workflow.\n\n## Execution environment\n\nThe execution environment is where an agent acts. It has four layers:"),
  Document(id='0e9efeb3-4549-41c3-a1b6-2681b634ed67', metadata={'source': 'https://docs.langchain.com/oss/python/deepagents/overview

# Retriver as a tool 

In [29]:
from langchain_core.tools import tool 

@tool 
def get_documents(query:str)->str: 
    """This tool will return the retrieved information"""
    docs = retriever.invoke(query)

    return "\n\n".join(doc.page_content for doc in docs)
    

## Add tool with standalone LLM

In [30]:
llm_with_rag_tool = primary_llm.bind_tools([get_documents])

In [31]:
llm_with_rag_tool

_ChatModelBinding(bound=ChatGroq(metadata={'lc_versions': {'langchain-core': '1.5.3', 'langchain': '1.3.14'}}, output_version=None, profile={'name': 'Llama 3.3 70B Versatile', 'release_date': '2024-12-06', 'last_updated': '2024-12-06', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 32768, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x124c320d0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x124c32ad0>, model_name='llama-3.3-70b-versatile', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None), kwargs={'tools': [{'type': 'function', 'function': {'name': 'get_documents', 'description': 'This tool will return t

In [32]:
from langchain.messages import HumanMessage, SystemMessage
result = llm_with_rag_tool.invoke(
    [
        SystemMessage(content="You are an helpful assistant"),
        HumanMessage(content="How to cretae a langchain agent assistant?")
    ]
)

tool = result.tool_calls[0]
tool

{'name': 'get_documents',
 'args': {'query': 'create Langchain agent assistant'},
 'id': 'fttj9nrf9',
 'type': 'tool_call'}

In [33]:
get_documents.invoke(tool)

ToolMessage(content="> ## Documentation Index\n> Fetch the complete documentation index at: https://docs.langchain.com/llms.txt\n> Use this file to discover all available pages before exploring further.\n\n# Quickstart\n\nThis quickstart demonstrates how to build a calculator agent using the LangGraph Graph API or the Functional API.\n\n<Tip>\n  **Using an AI coding assistant?**\n\n  * Install the [LangChain Docs MCP server](/use-these-docs) to give your agent access to up-to-date LangChain documentation and examples.\n  * Install [LangChain Skills](https://github.com/langchain-ai/langchain-skills) to improve your agent's performance on LangChain ecosystem tasks.\n</Tip>\n\n* [Use the Graph API](#use-the-graph-api) if you prefer to define your agent as a graph of nodes and edges.\n* [Use the Functional API](#use-the-functional-api) if you prefer to define your agent as a single function.\n\n> ## Documentation Index\n> Fetch the complete documentation index at: https://docs.langchain.co

## Add tool with Langchain Agent

In [38]:
from langchain.agents import create_agent

agent = create_agent(
    model=fallback_llm_1, 
    tools=[get_documents],
    system_prompt="""Use get documents tool for answering agent, deep agent, langchain and langgraph related queries."""
)

In [39]:
results = agent.invoke({
    "messages":[
        SystemMessage(content="You are an helpful assistant"),
                HumanMessage(content="How to cretae a langchain agent assistant?")
    ]
})

In [40]:
print(results["messages"][-1].content)

Here’s a practical way to create a **LangChain “agent assistant”** (tool-using chatbot) in Python. I’ll show the common pattern: **LLM + tools + agent + executor**, optionally with **memory**.

## 1) Install dependencies
```bash
pip install langchain langchain-openai langchain-community
```

## 2) Set up the LLM
Example with OpenAI:
```python
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
```

## 3) Create tools (functions the agent can call)
A tool is just a function wrapped with LangChain’s `@tool`.

```python
from langchain_core.tools import tool

@tool
def add(a: int, b: int) -> int:
    """Add two integers."""
    return a + b

@tool
def get_time() -> str:
    """Get the current time."""
    from datetime import datetime
    return datetime.now().isoformat()
```

## 4) Build the agent (ReAct-style)
Use LangChain’s agent creation utilities.

```python
from langchain.agents import create_tool_calling_agent, AgentExecutor
from langchain_